In [1]:
import gzip
import csv
import math
import io
import os
import re
import sys
import zipfile
import tempfile
import xml.etree.ElementTree as ET
from bisect import bisect_left
from pathlib import Path
from collections import deque

import pandas as pd

# ============================================================
# Convert filtered MATSim population to SUMO persons (multimodal + PT)
# ============================================================

# ============================================================
# PATHS (edit if needed)
# ============================================================
POP_PATH   = Path(r"../Eqasim_output/17population_filtered.xml.gz")
CSV_PATH   = Path(r"../2-POI's/facilities2sumo_multimode.csv")
STOPS_ADD  = Path(r"../3-public_transport/gtfs_pt_stops.add.xml")
VEHS_ADD   = Path(r"../3-public_transport/gtfs_pt_vehicles.add.xml")
NET_PATH   = Path(r"../1-network/cda_la_rochelle.net.xml")   # accepts .net.xml or .zip
GTFS_ZIP   = Path(r"../3-public_transport/input/ca_la_rochelle-aggregated-gtfs.zip")

OUT_ROU    = Path("population_all.rou.xml")
OUT_LOG    = Path("population_all.log.csv")

# ============================================================
# OPTIONS
# ============================================================
FORCE_LAST_STOP_TO_86400 = True
PRINT_EVERY = 100

# PT heuristics
K_EXPANSION_STEPS = (5, 10, 20, 50)
GLOBAL_ORIGIN_CANDIDATES = 30

# Walk time conversion
WALK_SPEED = 1.3      # m/s
COST_IS_TIME = False  # if True, shortest_cost() is already in seconds

# ---------- FAST PT (euclidean instead of Dijkstra) ----------
USE_EUCLID_FOR_PT = True
WALK_DETOUR = 1.30            # 1.2–1.5: increase if PT is too "optimistic"
MAX_WALK_PT_M = 30000         # safety: reject absurdly long PT walks

# Optional: check PT walk connectivity without Dijkstra (BFS component test)
PT_WALK_CONNECTIVITY_CHECK = True

# CONNECTIVITY PRECHECK (walk/car/bike)
ENABLE_CONNECTIVITY_PRECHECK = True
ENABLE_COMPONENT_DIAG = True
COMPONENT_BFS_MAX_VISITS = 250_000

# ============================================================
# SUMO / sumolib
# ============================================================
SUMO_HOME = os.environ.get("SUMO_HOME")
if not SUMO_HOME:
    raise RuntimeError(
        "SUMO_HOME is not set. Please set SUMO_HOME to your SUMO installation folder "
        "(it must contain the 'tools' directory)."
    )

tools_path = str(Path(SUMO_HOME) / "tools")
if tools_path not in sys.path:
    sys.path.append(tools_path)

import sumolib
from sumolib.geomhelper import positionAtShapeOffset
from math import hypot

def _extract_netxml_from_zip(zip_path: Path) -> Path:
    """Extract the first *.net.xml found in the zip into a temporary file."""
    with zipfile.ZipFile(zip_path, "r") as zf:
        net_members = [n for n in zf.namelist() if n.endswith(".net.xml")]
        if not net_members:
            raise ValueError(f"No .net.xml found in {zip_path}")
        member = net_members[0]
        tmpdir = Path(tempfile.mkdtemp(prefix="sumo_net_"))
        out = tmpdir / Path(member).name
        with zf.open(member) as src, out.open("wb") as dst:
            dst.write(src.read())
        return out

def load_net(path: Path):
    if path.suffix.lower() == ".zip":
        netxml = _extract_netxml_from_zip(path)
        return sumolib.net.readNet(str(netxml))
    return sumolib.net.readNet(str(path))

net = load_net(NET_PATH)

# ============================================================
# Helpers
# ============================================================
def ln(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def hms_to_seconds(hms: str) -> int:
    h, m, s = hms.strip().split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)

def fmt_pos(x: float) -> str:
    return f"{x:.2f}".rstrip("0").rstrip(".")

def is_nan_value(x) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        return math.isnan(x)
    s = str(x).strip().lower()
    return s in {"nan", "none", ""}

def is_interaction_activity(act_elem: ET.Element) -> bool:
    t = act_elem.get("type") or ""
    return "interaction" in t.lower()

def lane_to_edge(lane_id: str) -> str:
    return lane_id.rsplit("_", 1)[0] if "_" in lane_id else lane_id

# --- edge object cache ---
_edge_obj_cache = {}
def edge_obj(edge_id: str):
    if edge_id in _edge_obj_cache:
        return _edge_obj_cache[edge_id]
    e = net.getEdge(edge_id)
    if e is None:
        raise ValueError(f"unknown edge in net: {edge_id}")
    _edge_obj_cache[edge_id] = e
    return e

# --- euclidean helpers ---
def euclid2(ax, ay, bx, by):
    dx = ax - bx
    dy = ay - by
    return dx*dx + dy*dy

def dist_m(ax, ay, bx, by) -> float:
    return hypot(bx - ax, by - ay)

def walk_time_xy(ax, ay, bx, by) -> float:
    return (dist_m(ax, ay, bx, by) / WALK_SPEED) * WALK_DETOUR

# --- lane+pos -> x,y (for stops & facilities) ---
_lane_shape_cache = {}
_lane_pos_xy_cache = {}

def lane_shape(lane_id: str):
    lane_id = str(lane_id)
    if lane_id not in _lane_shape_cache:
        _lane_shape_cache[lane_id] = net.getLane(lane_id).getShape()
    return _lane_shape_cache[lane_id]

def lane_pos_xy(lane_id: str, pos: float):
    lane_id = str(lane_id)
    key = (lane_id, float(pos))
    if key in _lane_pos_xy_cache:
        return _lane_pos_xy_cache[key]
    x, y = positionAtShapeOffset(lane_shape(lane_id), float(pos))
    _lane_pos_xy_cache[key] = (float(x), float(y))
    return _lane_pos_xy_cache[key]

# --- shortest path cost cache (Dijkstra): walk/car/bike only ---
_sp_cost_cache = {}
def shortest_cost(from_edge: str, to_edge: str, vclass: str) -> float:
    key = (from_edge, to_edge, vclass)
    if key in _sp_cost_cache:
        return _sp_cost_cache[key]
    fe = edge_obj(from_edge)
    te = edge_obj(to_edge)
    path, cost = net.getShortestPath(fe, te, vClass=vclass)
    if path is None:
        _sp_cost_cache[key] = float("inf")
    else:
        _sp_cost_cache[key] = float(cost) if cost is not None else 0.0
    return _sp_cost_cache[key]

def has_route(from_edge: str, to_edge: str, vclass: str) -> bool:
    return math.isfinite(shortest_cost(from_edge, to_edge, vclass))

def walk_time_seconds(from_edge: str, to_edge: str) -> float:
    c = shortest_cost(from_edge, to_edge, "pedestrian")
    if not math.isfinite(c):
        return float("inf")
    return float(c) if COST_IS_TIME else float(c) / WALK_SPEED

# ============================================================
# Connectivity diagnostic: "disconnected components"
# ============================================================
_component_reach_cache = {}  # (from_edge, vclass) -> (visited_set or None, visited_count, reached_all?)

def _edge_allows_vclass(edge, vclass: str) -> bool:
    for ln_ in edge.getLanes():
        if ln_.allows(vclass):
            return True
    return False

def _neighbors_for_vclass(edge, vclass: str):
    for e2 in edge.getOutgoing().keys():
        if _edge_allows_vclass(e2, vclass):
            yield e2

def same_component_bfs(from_edge_id: str, to_edge_id: str, vclass: str) -> (bool, int):
    key = (from_edge_id, vclass)
    if key in _component_reach_cache:
        visited_set, visited_count, reached_all = _component_reach_cache[key]
        if visited_set is not None:
            return (to_edge_id in visited_set), visited_count

    start = edge_obj(from_edge_id)
    goal = to_edge_id

    q = deque([start])
    visited = set([from_edge_id])
    visits = 0

    while q:
        e = q.popleft()
        visits += 1
        if e.getID() == goal:
            _component_reach_cache[key] = (visited, len(visited), True)
            return True, len(visited)
        if visits > COMPONENT_BFS_MAX_VISITS:
            _component_reach_cache[key] = (None, visits, False)
            return False, visits
        for n in _neighbors_for_vclass(e, vclass):
            nid = n.getID()
            if nid not in visited:
                visited.add(nid)
                q.append(n)

    _component_reach_cache[key] = (visited, len(visited), True)
    return False, len(visited)

def connectivity_reason(from_edge: str, to_edge: str, vclass: str) -> str:
    if has_route(from_edge, to_edge, vclass):
        return ""
    if not ENABLE_COMPONENT_DIAG:
        return "no_route"
    ok, sz = same_component_bfs(from_edge, to_edge, vclass)
    if ok:
        return "no_route"
    return f"disconnected components (vclass={vclass}, component_visits~{sz})"

def pt_walk_is_possible_fast(from_edge: str, to_edge: str) -> bool:
    ok, _ = same_component_bfs(from_edge, to_edge, "pedestrian")
    return ok

# ============================================================
# facilities2sumo mapping (robust CSV parsing)
# ============================================================
def read_mapping_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Mapping CSV not found: {path.resolve()}")
    header = path.read_text(encoding="utf-8", errors="replace").splitlines()[0]
    if ";" in header and "," not in header:
        sep = ";"
    elif "," in header and ";" not in header:
        sep = ","
    else:
        sep = None
    df_ = pd.read_csv(path, sep=sep, engine="python", encoding="utf-8-sig")
    df_.columns = [str(c).strip().lstrip("\ufeff") for c in df_.columns]
    return df_

df = read_mapping_csv(CSV_PATH)

# If Step 5 wrote status/error_reason, keep only successful rows
if "status" in df.columns:
    df = df[df["status"] == "mapped"].copy()

required = {"poi_id", "mode", "edge_id", "lane_id", "pos"}
missing = required - set(df.columns)
if missing:
    raise ValueError(
        f"facilities2sumo_multimode.csv is missing columns: {missing}. "
        f"Available columns: {list(df.columns)}"
    )

fac_map = {(str(r.poi_id), str(r.mode)): (r.edge_id, r.lane_id, r.pos) for r in df.itertuples(index=False)}

def get_fac_sumo(poi_id: str, preferred_mode: str):
    for m in (preferred_mode, "passenger", "pedestrian"):
        k = (poi_id, m)
        if k in fac_map:
            edge, lane, pos = fac_map[k]
            if is_nan_value(edge) or is_nan_value(lane) or is_nan_value(pos):
                raise ValueError(f"nan mapping for facility={poi_id} mode={m} -> edge={edge}, lane={lane}, pos={pos}")
            return str(edge), str(lane), float(pos)

    for (p, _m), (edge, lane, pos) in fac_map.items():
        if p == poi_id:
            if is_nan_value(edge) or is_nan_value(lane) or is_nan_value(pos):
                raise ValueError(f"nan mapping for facility={poi_id} (fallback) -> edge={edge}, lane={lane}, pos={pos}")
            return str(edge), str(lane), float(pos)

    raise KeyError(f"Missing facility mapping for {poi_id!r}")

def get_fac_sumo_strict(poi_id: str, mode: str):
    k = (poi_id, mode)
    if k not in fac_map:
        raise KeyError(f"Missing strict facility mapping for {poi_id!r} mode={mode!r}")
    edge, lane, pos = fac_map[k]
    if is_nan_value(edge) or is_nan_value(lane) or is_nan_value(pos):
        raise ValueError(f"nan mapping for facility={poi_id} mode={mode} -> edge={edge}, lane={lane}, pos={pos}")
    return str(edge), str(lane), float(pos)

def fac_xy(poi_id: str, mode: str):
    e, l, p = get_fac_sumo(poi_id, mode)
    x, y = lane_pos_xy(l, p)
    return e, l, p, x, y

# ============================================================
# Parse PT stops.add.xml (stops_by_line)
# ============================================================
if not STOPS_ADD.exists():
    raise FileNotFoundError(f"PT stops file not found: {STOPS_ADD.resolve()}")

stops_by_line = {}
stop_index_by_line = {}
all_stops = []

st_root = ET.parse(STOPS_ADD).getroot()
for bs in st_root.iter():
    if ln(bs.tag) != "busStop":
        continue
    bs_id = bs.get("id")
    lane = bs.get("lane")
    if not bs_id or not lane:
        continue

    edge = lane_to_edge(lane)
    s = bs.get("startPos")
    e = bs.get("endPos")
    if s is None:
        continue
    try:
        s = float(s)
        pos = (s + float(e)) / 2.0 if e is not None else s
    except:
        continue

    line = bs_id.rsplit(".", 1)[0] if "." in bs_id else None
    if not line:
        continue

    # Exact stop coordinates from lane+pos
    sx, sy = lane_pos_xy(lane, pos)

    rec = {"busStopId": bs_id, "edge": edge, "lane": lane, "pos": pos, "x": sx, "y": sy}
    stops_by_line.setdefault(line, []).append(rec)
    all_stops.append(rec)

for line, lst in stops_by_line.items():
    stop_index_by_line[line] = {st["busStopId"]: i for i, st in enumerate(lst)}

def stop_index(line: str, stop_rec: dict) -> int:
    return stop_index_by_line[line].get(stop_rec["busStopId"], -1)

# ============================================================
# Parse vehicles.add.xml
# ============================================================
if not VEHS_ADD.exists():
    raise FileNotFoundError(f"PT vehicles file not found: {VEHS_ADD.resolve()}")

available_lines = set()
depart_starts_by_line = {}
matsim_route_to_line = {}

veh_root = ET.parse(VEHS_ADD).getroot()
for v in veh_root.iter():
    if ln(v.tag) != "vehicle":
        continue
    line = v.get("line")
    vid = v.get("id")
    dep = v.get("depart")
    if line:
        available_lines.add(line)
        if dep is not None:
            try:
                depart_starts_by_line.setdefault(line, []).append(float(dep))
            except:
                pass

    # Map base vehicle id prefix -> line (heuristic)
    if line and vid and "." in vid:
        base = vid.rsplit(".", 1)[0]
        matsim_route_to_line.setdefault(base, line)

for k in depart_starts_by_line:
    depart_starts_by_line[k].sort()

# ============================================================
# Load GTFS stop_times offsets (optional, improves PT time feasibility)
# ============================================================
line_stop_offsets = {}

if GTFS_ZIP.exists():
    with zipfile.ZipFile(GTFS_ZIP) as zf:
        with zf.open("stop_times.txt") as f:
            st = pd.read_csv(f, low_memory=False)

    st = st[["trip_id", "departure_time", "stop_sequence"]].copy()
    st["stop_sequence"] = pd.to_numeric(st["stop_sequence"], errors="coerce")
    st = st.dropna(subset=["trip_id", "departure_time", "stop_sequence"])
    st["stop_sequence"] = st["stop_sequence"].astype(int)

    for trip_id, g in st.groupby("trip_id"):
        g = g.sort_values("stop_sequence")
        times = []
        ok = True
        for t in g["departure_time"].astype(str).tolist():
            try:
                times.append(hms_to_seconds(t))
            except:
                ok = False
                break
        if not ok or not times:
            continue
        t0 = times[0]
        offsets = [ti - t0 for ti in times]
        line = matsim_route_to_line.get(trip_id)
        if line:
            line_stop_offsets.setdefault(line, offsets)
else:
    print("[WARN] GTFS_ZIP not found: PT time filtering will be less precise (but the pipeline can still run).")

def next_depart_start_for_board(line: str, board_idx: int, t_arrive_board: float):
    departs = depart_starts_by_line.get(line)
    if not departs:
        return None, float("inf")

    offsets = line_stop_offsets.get(line)
    off = 0.0
    if offsets and 0 <= board_idx < len(offsets):
        off = float(offsets[board_idx])

    threshold = t_arrive_board - off
    i = bisect_left(departs, threshold)
    if i >= len(departs):
        return None, float("inf")

    d0 = float(departs[i])
    return d0, d0 + off

def bus_time_at_stop(line: str, depart_start: float, stop_idx: int) -> float:
    offsets = line_stop_offsets.get(line)
    if offsets and 0 <= stop_idx < len(offsets):
        return depart_start + float(offsets[stop_idx])
    return depart_start

# ============================================================
# MATSim parsing
# ============================================================
def get_selected_plan(person_elem: ET.Element):
    plans = [ch for ch in list(person_elem) if ln(ch.tag) == "plan"]
    if not plans:
        return None
    for p in plans:
        if p.get("selected") == "yes":
            return p
    return plans[0]

def get_child_attr_value(elem: ET.Element, attr_name: str):
    for ch in elem:
        if ln(ch.tag) == "attributes":
            for a in ch:
                if ln(a.tag) == "attribute" and a.get("name") == attr_name:
                    return (a.text or "").strip()
    return None

def extract_route_json(leg: ET.Element) -> str:
    for ch in leg:
        if ln(ch.tag) == "route" and ch.text:
            return ch.text
    return ""

def extract_transit_route_id_from_leg(pt_leg: ET.Element):
    txt = extract_route_json(pt_leg)
    m = re.search(r'"transitRouteId"\s*:\s*"([^"]+)"', txt)
    return m.group(1) if m else None

def build_activity_leg_sequence(plan: ET.Element):
    children = list(plan)
    kept_idx = [i for i, ch in enumerate(children)
                if ln(ch.tag) == "activity" and not is_interaction_activity(ch)]
    acts = [children[i] for i in kept_idx]
    if not acts:
        return [], []

    legs_between = []
    for a_i, idx in enumerate(kept_idx[:-1]):
        nxt = kept_idx[a_i + 1]
        legs = [ch for ch in children[idx+1:nxt] if ln(ch.tag) == "leg"]
        legs_between.append(legs)

    return acts, legs_between

def merge_same_facility(acts, legs_between):
    merged = []
    group_starts = []
    for i, act in enumerate(acts):
        if not merged:
            merged.append(act)
            group_starts.append(i)
            continue
        if act.get("facility") == merged[-1].get("facility"):
            if act.get("end_time") is not None:
                merged[-1].set("end_time", act.get("end_time"))
            if (merged[-1].get("type") in (None, "", "unknown")) and act.get("type"):
                merged[-1].set("type", act.get("type"))
        else:
            merged.append(act)
            group_starts.append(i)

    merged_legs_between = []
    for g in range(len(merged) - 1):
        start_idx = group_starts[g]
        end_idx = group_starts[g + 1]
        agg = []
        for k in range(start_idx, end_idx):
            agg.extend(legs_between[k])
        merged_legs_between.append(agg)

    return merged, merged_legs_between

def classify_car_role(legs):
    role = None
    for leg in legs:
        mode = leg.get("mode", "walk")
        routing = get_child_attr_value(leg, "routingMode")
        if mode in ("car", "car_passenger") or routing in ("car", "car_passenger"):
            this = "car_passenger" if (mode == "car_passenger" or routing == "car_passenger") else "car"
            if role != "car_passenger":
                role = this
    return role

def has_any_pt(legs):
    for leg in legs:
        mode = leg.get("mode", "walk")
        routing = get_child_attr_value(leg, "routingMode")
        if mode == "pt" or routing == "pt":
            return True
    return False

def collect_pt_route_ids(legs):
    routes = []
    for leg in legs:
        if leg.get("mode") == "pt":
            rid = extract_transit_route_id_from_leg(leg)
            if rid:
                routes.append(rid)
    return routes

def has_any_bike(legs):
    for leg in legs:
        mode = (leg.get("mode") or "").lower()
        routing = (get_child_attr_value(leg, "routingMode") or "").lower()
        if mode in ("bike", "bicycle") or routing in ("bike", "bicycle"):
            return True
    return False

# ============================================================
# PT selection (ORDER + TIME) — FAST option
# ============================================================
def choose_board_alight_on_line_xy(line: str, ox: float, oy: float, tx: float, ty: float, t_depart: float):
    if line not in stops_by_line:
        raise ValueError(f"line has no stops: {line}")
    vj_stops = stops_by_line[line]
    if len(vj_stops) < 2:
        raise ValueError(f"line too short: {line}")

    for K in K_EXPANSION_STEPS:
        board_candidates = sorted(vj_stops, key=lambda st: euclid2(ox, oy, st["x"], st["y"]))[:min(K, len(vj_stops))]

        for board in board_candidates:
            b_idx = stop_index(line, board)
            if b_idx < 0 or b_idx >= len(vj_stops) - 1:
                continue

            after = vj_stops[b_idx + 1:]
            if not after:
                continue

            for K2 in K_EXPANSION_STEPS:
                alight_candidates = sorted(after, key=lambda st: euclid2(tx, ty, st["x"], st["y"]))[:min(K2, len(after))]
                if not alight_candidates:
                    continue
                best_alight = min(alight_candidates, key=lambda st: euclid2(tx, ty, st["x"], st["y"]))

                # Time feasibility (euclidean)
                if USE_EUCLID_FOR_PT:
                    walk_m = dist_m(ox, oy, board["x"], board["y"])
                    if walk_m > MAX_WALK_PT_M:
                        continue
                    t_arrive_board = t_depart + walk_time_xy(ox, oy, board["x"], board["y"])
                else:
                    t_arrive_board = t_depart

                d0, _t_bus_board = next_depart_start_for_board(line, b_idx, t_arrive_board)
                if d0 is None:
                    continue

                # Optional: PT walk connectivity check without Dijkstra
                if PT_WALK_CONNECTIVITY_CHECK:
                    if not pt_walk_is_possible_fast(board["edge"], best_alight["edge"]):
                        continue

                return board, best_alight, d0

    raise ValueError(f"no ordered+time PT on {line}")

def choose_best_line_global_xy(ox: float, oy: float, tx: float, ty: float, t_depart: float):
    candidates = sorted(all_stops, key=lambda st: euclid2(ox, oy, st["x"], st["y"]))[:min(GLOBAL_ORIGIN_CANDIDATES, len(all_stops))]

    best = None
    best_total = float("inf")

    for st_from in candidates:
        line = st_from["busStopId"].rsplit(".", 1)[0]
        if line not in stops_by_line:
            continue
        if line not in available_lines:
            continue

        try:
            board, alight, d0 = choose_board_alight_on_line_xy(line, ox, oy, tx, ty, t_depart)
        except Exception:
            continue

        total = euclid2(ox, oy, board["x"], board["y"]) + euclid2(alight["x"], alight["y"], tx, ty)
        if total < best_total:
            best_total = total
            best = (line, board, alight, d0)

    if best is None:
        raise ValueError(f"no PT line found (euclid) at t={t_depart}")
    return best

def choose_two_rides_xy(line1: str, line2: str, ox: float, oy: float, tx: float, ty: float, t_depart: float):
    if line1 not in stops_by_line or line2 not in stops_by_line:
        raise ValueError("missing stops for line1/line2")

    l1 = stops_by_line[line1]
    l2 = stops_by_line[line2]
    if len(l1) < 2 or len(l2) < 2:
        raise ValueError("line too short")

    l2_pts = [(st["x"], st["y"]) for st in l2]

    def dist_to_line2(st):
        sx, sy = st["x"], st["y"]
        return min((sx-x)*(sx-x) + (sy-y)*(sy-y) for x, y in l2_pts) if l2_pts else 0.0

    hubs = sorted(l1, key=dist_to_line2)[:min(25, len(l1))]

    best = None
    best_total = float("inf")

    for hub in hubs:
        try:
            board1, alight1, d01 = choose_board_alight_on_line_xy(line1, ox, oy, hub["x"], hub["y"], t_depart)
        except Exception:
            continue

        a1_idx = stop_index(line1, alight1)
        t_alight1 = bus_time_at_stop(line1, d01, a1_idx)

        board2_candidates = sorted(l2, key=lambda st: euclid2(alight1["x"], alight1["y"], st["x"], st["y"]))[:min(15, len(l2))]

        for b2 in board2_candidates:
            b2_idx = stop_index(line2, b2)
            if b2_idx < 0 or b2_idx >= len(l2) - 1:
                continue

            # Transfer walk time (euclidean)
            transfer_m = dist_m(alight1["x"], alight1["y"], b2["x"], b2["y"])
            if transfer_m > MAX_WALK_PT_M:
                continue
            t_arrive_b2 = t_alight1 + walk_time_xy(alight1["x"], alight1["y"], b2["x"], b2["y"])

            after = l2[b2_idx+1:]
            if not after:
                continue
            alight2 = min(after, key=lambda st: euclid2(tx, ty, st["x"], st["y"]))

            d02, _t_bus_b2 = next_depart_start_for_board(line2, b2_idx, t_arrive_b2)
            if d02 is None:
                continue

            access = walk_time_xy(ox, oy, board1["x"], board1["y"])
            egress = walk_time_xy(alight2["x"], alight2["y"], tx, ty)
            total = access + walk_time_xy(alight1["x"], alight1["y"], b2["x"], b2["y"]) + egress

            if total < best_total:
                best_total = total
                best = {
                    "board1": board1, "alight1": alight1, "d01": d01,
                    "board2": b2, "alight2": alight2, "d02": d02,
                    "line1": line1, "line2": line2
                }

    if best is None:
        raise ValueError("no feasible 2-ride PT chain found (euclid)")
    return best

# ============================================================
# Writer (walk/car/bike/pt)
# ============================================================
def incoming_pref(seg_kind: str) -> str:
    return "passenger" if seg_kind == "car" else "pedestrian"

def choose_bike_edges_and_pos(from_fac: str, to_fac: str):
    fe_p, _fl_p, fp_p = get_fac_sumo(from_fac, "pedestrian")
    te_p, _tl_p, tp_p = get_fac_sumo(to_fac, "pedestrian")
    edge_obj(fe_p); edge_obj(te_p)

    candidates = [((fe_p, fp_p), (te_p, tp_p), "ped,ped")]

    fe_b = te_b = None
    try:
        fe_b, _fl_b, fp_b = get_fac_sumo_strict(from_fac, "bicycle")
        edge_obj(fe_b)
    except Exception:
        fe_b = None

    try:
        te_b, _tl_b, tp_b = get_fac_sumo_strict(to_fac, "bicycle")
        edge_obj(te_b)
    except Exception:
        te_b = None

    if te_b is not None:
        candidates.append(((fe_p, fp_p), (te_b, tp_b), "ped,bike"))
    if fe_b is not None:
        candidates.append(((fe_b, fp_b), (te_p, tp_p), "bike,ped"))
    if fe_b is not None and te_b is not None:
        candidates.append(((fe_b, fp_b), (te_b, tp_b), "bike,bike"))

    for (fe, fp), (te, tp), _tag in candidates:
        if has_route(fe, te, "bicycle"):
            return fe, fp, te, tp

    reason = connectivity_reason(fe_p, te_p, "bicycle")
    if reason:
        raise ValueError(f"no bicycle route {fe_p}->{te_p} ({reason})")
    raise ValueError(f"no bicycle route {fe_p}->{te_p}")

def write_person_to_string(person_elem: ET.Element) -> str:
    pid = person_elem.get("id", "")
    plan = get_selected_plan(person_elem)
    if plan is None:
        raise ValueError("no plan")

    acts, legs_between = build_activity_leg_sequence(plan)
    if not acts:
        raise ValueError("no non-interaction activities")

    acts, legs_between = merge_same_facility(acts, legs_between)

    # Priority: car > pt > bike > walk
    seg_kinds = []
    seg_pt_routes = []
    for legs in legs_between:
        car_role = classify_car_role(legs)
        if car_role is not None:
            seg_kinds.append(("car", car_role))
            seg_pt_routes.append([])
            continue

        if has_any_pt(legs):
            seg_kinds.append(("pt", None))
            seg_pt_routes.append(collect_pt_route_ids(legs))
            continue

        if has_any_bike(legs):
            seg_kinds.append(("bike", None))
            seg_pt_routes.append([])
            continue

        seg_kinds.append(("walk", None))
        seg_pt_routes.append([])

    # Connectivity pre-check (excluding PT)
    if ENABLE_CONNECTIVITY_PRECHECK:
        for i in range(len(acts) - 1):
            fac_a = acts[i].get("facility")
            fac_b = acts[i+1].get("facility")
            kind, extra = seg_kinds[i]
            if not fac_a or not fac_b:
                continue

            if kind == "walk":
                ea, _, _ = get_fac_sumo(fac_a, "pedestrian")
                eb, _, _ = get_fac_sumo(fac_b, "pedestrian")
                edge_obj(ea); edge_obj(eb)
                if not has_route(ea, eb, "pedestrian"):
                    reason = connectivity_reason(ea, eb, "pedestrian")
                    raise ValueError(f"no pedestrian route {ea}->{eb} ({reason})")

            elif kind == "car":
                ea, _, _ = get_fac_sumo(fac_a, "passenger")
                eb, _, _ = get_fac_sumo(fac_b, "passenger")
                edge_obj(ea); edge_obj(eb)
                if not has_route(ea, eb, "passenger"):
                    reason = connectivity_reason(ea, eb, "passenger")
                    raise ValueError(f"no car route {ea}->{eb} ({reason})")

            elif kind == "bike":
                _ = choose_bike_edges_and_pos(fac_a, fac_b)

    buf = io.StringIO()
    buf.write(f'    <person id="{pid}" depart="0">\n')

    for i, act in enumerate(acts):
        act_type = act.get("type") or "unknown"
        facility = act.get("facility")
        if not facility:
            raise ValueError("activity missing facility")

        end_time = act.get("end_time")
        until_sec = hms_to_seconds(end_time) if end_time else 86400
        if FORCE_LAST_STOP_TO_86400 and i == len(acts) - 1:
            until_sec = 86400

        if i == 0:
            arr_kind = seg_kinds[0][0] if seg_kinds else "walk"
        else:
            arr_kind = seg_kinds[i-1][0]

        st_edge, st_lane, st_pos = get_fac_sumo(facility, incoming_pref(arr_kind))
        edge_obj(st_edge)

        buf.write(f'        <stop lane="{st_lane}" startPos="{fmt_pos(st_pos)}" until="{until_sec}" actType="{act_type}" />\n')

        if i == len(acts) - 1:
            break

        next_fac = acts[i+1].get("facility")
        if not next_fac:
            raise ValueError("next activity missing facility")

        kind, extra = seg_kinds[i]
        t_depart = float(until_sec)

        # ---------------- WALK ----------------
        if kind == "walk":
            dep_edge, _, dep_pos = get_fac_sumo(facility, "pedestrian")
            dst_edge, _, dst_pos = get_fac_sumo(next_fac, "pedestrian")
            edge_obj(dep_edge); edge_obj(dst_edge)

            if st_edge != dep_edge or abs(st_pos - dep_pos) > 1e-6:
                if has_route(st_edge, dep_edge, "pedestrian"):
                    buf.write(f'        <walk from="{st_edge}" to="{dep_edge}" departPos="{fmt_pos(st_pos)}" arrivalPos="{fmt_pos(dep_pos)}" />\n')
                else:
                    dep_edge, dep_pos = st_edge, st_pos

            if not has_route(dep_edge, dst_edge, "pedestrian"):
                reason = connectivity_reason(dep_edge, dst_edge, "pedestrian")
                raise ValueError(f"no pedestrian route {dep_edge}->{dst_edge} ({reason})")

            buf.write(f'        <walk from="{dep_edge}" to="{dst_edge}" departPos="{fmt_pos(dep_pos)}" arrivalPos="{fmt_pos(dst_pos)}" />\n')

        # ---------------- CAR ----------------
        elif kind == "car":
            car_role = extra or "car"
            dep_edge, _, dep_pos = get_fac_sumo(facility, "passenger")
            dst_edge, _, dst_pos = get_fac_sumo(next_fac, "passenger")
            edge_obj(dep_edge); edge_obj(dst_edge)

            if st_edge != dep_edge or abs(st_pos - dep_pos) > 1e-6:
                if has_route(st_edge, dep_edge, "pedestrian"):
                    buf.write(f'        <walk from="{st_edge}" to="{dep_edge}" departPos="{fmt_pos(st_pos)}" arrivalPos="{fmt_pos(dep_pos)}" />\n')
                else:
                    dep_edge, dep_pos = st_edge, st_pos

            if not has_route(dep_edge, dst_edge, "passenger"):
                reason = connectivity_reason(dep_edge, dst_edge, "passenger")
                raise ValueError(f"no car route {dep_edge}->{dst_edge} ({reason})")

            buf.write(f'        <personTrip from="{dep_edge}" to="{dst_edge}" departPos="{fmt_pos(dep_pos)}" arrivalPos="{fmt_pos(dst_pos)}" modes="car" role="{car_role}" />\n')

        # ---------------- BIKE ----------------
        elif kind == "bike":
            dep_edge, dep_pos, dst_edge, dst_pos = choose_bike_edges_and_pos(facility, next_fac)
            edge_obj(dep_edge); edge_obj(dst_edge)

            if st_edge != dep_edge or abs(st_pos - dep_pos) > 1e-6:
                if has_route(st_edge, dep_edge, "pedestrian"):
                    buf.write(f'        <walk from="{st_edge}" to="{dep_edge}" departPos="{fmt_pos(st_pos)}" arrivalPos="{fmt_pos(dep_pos)}" />\n')
                else:
                    dep_edge, dep_pos = st_edge, st_pos

            if not has_route(dep_edge, dst_edge, "bicycle"):
                reason = connectivity_reason(dep_edge, dst_edge, "bicycle")
                raise ValueError(f"no bicycle route {dep_edge}->{dst_edge} ({reason})")

            buf.write(f'        <personTrip from="{dep_edge}" to="{dst_edge}" departPos="{fmt_pos(dep_pos)}" arrivalPos="{fmt_pos(dst_pos)}" modes="bicycle" />\n')

        # ---------------- PT (FAST) ----------------
        elif kind == "pt":
            act_edge_p, act_lane_p, act_pos_p = get_fac_sumo(facility, "pedestrian")
            nxt_edge_p, nxt_lane_p, nxt_pos_p = get_fac_sumo(next_fac, "pedestrian")
            edge_obj(act_edge_p); edge_obj(nxt_edge_p)

            # Exact facility coordinates (lane+pos)
            act_x, act_y = lane_pos_xy(act_lane_p, act_pos_p)
            nxt_x, nxt_y = lane_pos_xy(nxt_lane_p, nxt_pos_p)

            if st_edge != act_edge_p or abs(st_pos - act_pos_p) > 1e-6:
                if has_route(st_edge, act_edge_p, "pedestrian"):
                    buf.write(f'        <walk from="{st_edge}" to="{act_edge_p}" departPos="{fmt_pos(st_pos)}" arrivalPos="{fmt_pos(act_pos_p)}" />\n')
                else:
                    act_edge_p, act_pos_p = st_edge, st_pos
                    # act_x/act_y remain approximate, but acceptable for fallback

            routes = seg_pt_routes[i]
            preferred_lines = []
            for rid in routes:
                line = matsim_route_to_line.get(rid) or (rid if rid in available_lines else None)
                if line:
                    preferred_lines.append(line)

            if len(preferred_lines) >= 2:
                line1, line2 = preferred_lines[0], preferred_lines[1]
                try:
                    chain = choose_two_rides_xy(line1, line2, act_x, act_y, nxt_x, nxt_y, t_depart)

                    b1 = chain["board1"]; a1 = chain["alight1"]
                    buf.write(f'        <walk from="{act_edge_p}" to="{b1["edge"]}" departPos="{fmt_pos(act_pos_p)}" arrivalPos="{fmt_pos(b1["pos"])}" />\n')
                    buf.write(f'        <ride from="{b1["edge"]}" to="{a1["edge"]}" lines="{chain["line1"]}" />\n')

                    b2 = chain["board2"]; a2 = chain["alight2"]
                    buf.write(f'        <walk from="{a1["edge"]}" to="{b2["edge"]}" departPos="{fmt_pos(a1["pos"])}" arrivalPos="{fmt_pos(b2["pos"])}" />\n')
                    buf.write(f'        <ride from="{b2["edge"]}" to="{a2["edge"]}" lines="{chain["line2"]}" />\n')

                    buf.write(f'        <walk from="{a2["edge"]}" to="{nxt_edge_p}" departPos="{fmt_pos(a2["pos"])}" arrivalPos="{fmt_pos(nxt_pos_p)}" />\n')

                except Exception:
                    line, board, alight, _d0 = choose_best_line_global_xy(act_x, act_y, nxt_x, nxt_y, t_depart)
                    buf.write(f'        <walk from="{act_edge_p}" to="{board["edge"]}" departPos="{fmt_pos(act_pos_p)}" arrivalPos="{fmt_pos(board["pos"])}" />\n')
                    buf.write(f'        <ride from="{board["edge"]}" to="{alight["edge"]}" lines="{line}" />\n')
                    buf.write(f'        <walk from="{alight["edge"]}" to="{nxt_edge_p}" departPos="{fmt_pos(alight["pos"])}" arrivalPos="{fmt_pos(nxt_pos_p)}" />\n')

            else:
                chosen = None
                if preferred_lines:
                    line_try = preferred_lines[0]
                    try:
                        board, alight, _d0 = choose_board_alight_on_line_xy(line_try, act_x, act_y, nxt_x, nxt_y, t_depart)
                        chosen = (line_try, board, alight)
                    except Exception:
                        chosen = None

                if chosen is None:
                    line, board, alight, _d0 = choose_best_line_global_xy(act_x, act_y, nxt_x, nxt_y, t_depart)
                else:
                    line, board, alight = chosen

                buf.write(f'        <walk from="{act_edge_p}" to="{board["edge"]}" departPos="{fmt_pos(act_pos_p)}" arrivalPos="{fmt_pos(board["pos"])}" />\n')
                buf.write(f'        <ride from="{board["edge"]}" to="{alight["edge"]}" lines="{line}" />\n')
                buf.write(f'        <walk from="{alight["edge"]}" to="{nxt_edge_p}" departPos="{fmt_pos(alight["pos"])}" arrivalPos="{fmt_pos(nxt_pos_p)}" />\n')

        else:
            raise ValueError(f"unknown segment kind: {kind}")

    buf.write("    </person>\n")
    return buf.getvalue()

# ============================================================
# RUN
# ============================================================
total = written = skipped = 0
skipped_nan = skipped_no_route = skipped_unknown_edge = 0

with OUT_LOG.open("w", newline="", encoding="utf-8") as log_f, OUT_ROU.open("w", encoding="utf-8") as out_f:
    log = csv.writer(log_f)
    log.writerow(["person_id", "status", "reason"])

    out_f.write('<?xml version="1.0" encoding="UTF-8"?>\n<routes>\n')

    with gzip.open(POP_PATH, "rb") as f:
        for event, elem in ET.iterparse(f, events=("end",)):
            if ln(elem.tag) == "person":
                total += 1
                pid = elem.get("id", "")

                try:
                    out_f.write(write_person_to_string(elem))
                    written += 1
                    log.writerow([pid, "ok", ""])
                except Exception as e:
                    skipped += 1
                    msg = str(e).lower()

                    if "nan mapping" in msg or "nan" in msg:
                        skipped_nan += 1
                        log.writerow([pid, "skipped_nan", str(e)])

                    elif ("no pedestrian route" in msg) or ("no car route" in msg) or ("no bicycle route" in msg) \
                         or ("no pt line found" in msg) or ("no ordered+time" in msg) or ("no feasible 2-ride" in msg):
                        skipped_no_route += 1
                        log.writerow([pid, "skipped_no_route", str(e)])

                    elif "unknown edge in net" in msg:
                        skipped_unknown_edge += 1
                        log.writerow([pid, "skipped_unknown_edge", str(e)])

                    else:
                        log.writerow([pid, "skipped_other", str(e)])

                elem.clear()

                if total % PRINT_EVERY == 0:
                    print(f"[progress] total={total} written={written} skipped={skipped} (nan={skipped_nan}, noroute={skipped_no_route})")

    out_f.write("</routes>\n")

print("\n[OK] DONE")
print(f"total={total}, written={written}, skipped={skipped}")
print(f"  - skipped_nan         : {skipped_nan}")
print(f"  - skipped_no_route    : {skipped_no_route}")
print(f"  - skipped_unknown_edge: {skipped_unknown_edge}")
print(f"Routes: {OUT_ROU.resolve()}")
print(f"Log   : {OUT_LOG.resolve()}")
print(f"K steps (PT): {K_EXPANSION_STEPS}")
print(f"PT lines: {len(available_lines)}")
print(f"routeId->line map: {len(matsim_route_to_line)}")

[progress] total=100 written=100 skipped=0 (nan=0, noroute=0)
[progress] total=200 written=196 skipped=4 (nan=0, noroute=4)
[progress] total=300 written=291 skipped=9 (nan=0, noroute=9)
[progress] total=400 written=390 skipped=10 (nan=0, noroute=10)
[progress] total=500 written=487 skipped=13 (nan=0, noroute=13)
[progress] total=600 written=582 skipped=18 (nan=0, noroute=18)
[progress] total=700 written=679 skipped=21 (nan=0, noroute=20)
[progress] total=800 written=776 skipped=24 (nan=0, noroute=23)
[progress] total=900 written=876 skipped=24 (nan=0, noroute=23)
[progress] total=1000 written=975 skipped=25 (nan=0, noroute=24)
[progress] total=1100 written=1073 skipped=27 (nan=0, noroute=26)
[progress] total=1200 written=1171 skipped=29 (nan=0, noroute=28)
[progress] total=1300 written=1265 skipped=35 (nan=0, noroute=34)
[progress] total=1400 written=1364 skipped=36 (nan=0, noroute=35)
[progress] total=1500 written=1463 skipped=37 (nan=0, noroute=36)
[progress] total=1600 written=1562 